# Control 03 · K00 exact legacy A/A · memmap A/A v2

角色：`exact_legacy_baseline_control`；证据等级：`CONTROL`。

本文件保留 Baseline 原始 StockTransformer 模型块和原始实例化路径；它是 A/A 控制，不是新的 alpha 候选。

数据、标签、memmap、训练统计量、验证选择器和最终
`date,instrument,score` 契约继承 `Baseline_memmap_strict_AA.ipynb`。

历史本地指标只证明 Goal0725 对应 sealed run；本 Notebook 使用 Baseline 的
`2019-01-01~2023-06-30 / 2023-07-01~2023-12-31` 切分重新训练，因此在真实平台
完整执行前，不把历史指标写成该导出文件的已复现效果。平台仍会执行完整 BARRA
风格剔除；本地五代理不能替代官方评测。


In [ ]:
def main(datasources, start_date, end_date):
    """K00 的 memmap 严格 A/A 平台实现。

    平台只替换 datasources / start_date / end_date。训练、验证、模型、标签、
    随机种子、优化器和输出口径均对齐 0721Pre_Deal/Baseline.ipynb；唯一有意改变的
    工程路径是将未标准化窗口连续写入磁盘 memmap，并在 DataLoader collate 阶段
    对当前 batch 应用训练集统计量。

    A/A 对齐原则：
    1. BASE10 字段、字段顺序、40 日历史缓冲、instrument 顺序和样本顺序对齐 Baseline；
    2. 标签、输入 log/fillna 语义以及训练集 1%/99% 标签截尾对齐 Baseline；
    3. 直接在完整 memmap 视图上调用 Baseline 同式的 float32 mean/std 归约；
    4. 保留 Baseline 的 seed、CUDA 初始化、DataLoader shuffle 和训练循环语义；
    5. manifest 保存 X/y/key/stats 哈希，供内存版逐项审计。
    """
    import time
    import random
    import copy
    import os
    import math
    import json
    import hashlib
    import shutil
    import gc
    from collections import deque
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import dai

    # 线上执行器可能通过 fork/worker 运行 main。必须在 import torch 前设置，
    # 使 CUDA 可用性探测优先走 NVML，避免提前初始化 CUDA runtime。
    os.environ.setdefault("PYTORCH_NVML_BASED_CUDA_CHECK", "1")

    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    import structlog

    logger = structlog.get_logger()

    EXPERIMENT_CONFIG = {
        "seq_len": 64,
        "d_model": 128,
        "nhead": 4,
        "nlayers": 3,
        "dim_ff": 256,
        "encoder_dropout": 0.1,
        "head_dropout": 0.0,
        "input_field_dropout": 0.0,
        "gaussian_noise": 0.0,
        "rdrop_lambda": 0.0,
        "pooling": "mean",
        "causal_attention": False,
        "pos_encoding": "sinusoidal",
        "head_type": "linear",
        "seed": 42,
        "epochs": 10,
        "batch_size": 512,
        "lr": 0.001,
        "patience": 3,
        "min_delta": 0.0,
        "early_stop_metric": "official_ic_plus_ir",
        "experiment_id": "TOP5_05_K00_memmap_strict_AA",
        "price_experiment": "N0",
        "model_name": 'baseline_transformer',
        "model_kwargs": {'d_model': 128, 'nhead': 4, 'layers': 3, 'dim_feedforward': 256, 'dropout': 0.1, 'max_len': 512},
        "expected_parameters": 399233,
        "model_only_delta_from_baseline": False,
        "strict_baseline_control": True,
    }

    DATA_CONFIG = {
        "field_set": "BASE10",
        # 与 0721 Baseline 的 INSTRUMENT_CHUNK_SIZE 完全一致，保持查询及写盘顺序。
        "instrument_chunk_size": 20,
        # 仅控制写盘峰值内存，不改变样本顺序或内容。
        "write_buffer_samples": 8192,
        # Baseline 公式在 SEQ_LEN=64 时得到 40 个日历日。
        "buffer_days": 40,
        # num_workers=0 保持和 Baseline 主进程 DataLoader 一致。
        "num_workers": 0,
        "prefetch_factor": 1,
        "rebuild_cache": True,
        "cleanup_cache_at_end": False,
        "cache_root": "./bigalpha_memmap_cache_top5_05_k00_strict_aa",
    }

    # ---------- 配置 ----------
    TRAIN_TABLE = "bigalpha_2026_stock_bar1m"
    INFER_TABLE = datasources["bar1m"]
    TRAIN_START, TRAIN_END = "2019-01-01", "2023-06-30 23:59:59"
    VALID_START, VALID_END = "2023-07-01", "2023-12-31 23:59:59"

    SEQ_LEN = int(EXPERIMENT_CONFIG["seq_len"])
    EPOCHS = int(EXPERIMENT_CONFIG["epochs"])
    BATCH = int(EXPERIMENT_CONFIG["batch_size"])
    LR = float(EXPERIMENT_CONFIG["lr"])
    SEED = int(EXPERIMENT_CONFIG["seed"])
    PATIENCE = int(EXPERIMENT_CONFIG["patience"])
    MIN_DELTA = float(EXPERIMENT_CONFIG["min_delta"])
    MAX_TRAIN_INSTRUMENTS = None

    FIELD_SET = DATA_CONFIG["field_set"]
    PRICE_COLS = ["open", "high", "low", "close", "bid_price1", "ask_price1"]
    VOL_COLS = ["volume", "amount", "bid_volume1", "ask_volume1"]
    FEATURE_COLS = PRICE_COLS + VOL_COLS
    N_FEAT = len(FEATURE_COLS)
    assert N_FEAT == 10, f"输入字段数量发生变化: {N_FEAT}"

    random.seed(SEED)
    np.random.seed(SEED)

    # 只设置 CPU 默认生成器；此处禁止 torch.manual_seed/manual_seed_all，
    # 因为它们会涉及 CUDA RNG。线上 DAI/runner 可能随后 fork，提前触碰
    # CUDA runtime 会造成只在线上出现的无效 CUDA 上下文。
    _cpu_generator = torch.Generator(device="cpu")
    _cpu_generator.manual_seed(SEED)
    torch.set_rng_state(_cpu_generator.get_state())
    del _cpu_generator

    # CUDA 初始化推迟到所有 DAI 取数和 memmap 构建完成之后。
    device = None
    cache_root = Path(DATA_CONFIG["cache_root"]).resolve()

    # 先合并为单一字典，再传给 structlog。
    # 不可在函数调用中同时显式传入 field_set/cache_root，
    # 又通过 **DATA_CONFIG 重复展开同名关键字。
    log_config = {
        **EXPERIMENT_CONFIG,
        **DATA_CONFIG,
        "device": "deferred_until_after_DAI",
        "torch_version": torch.__version__,
        "torch_cuda_build": torch.version.cuda,
        "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
        "pytorch_nvml_based_cuda_check": os.environ.get("PYTORCH_NVML_BASED_CUDA_CHECK"),
        "field_set": FIELD_SET,
        "feature_cols": FEATURE_COLS,
        "n_feat": N_FEAT,
        "price_preprocessing": "价格原值 + 训练集全局 z-score（控制组）",
        "baseline_source_sha256": "c0b41f3bf863055581e6ab52ae7cd7ea0d52d4a9871888252f6d17230e10fecb",
        "candidate_id": 'K00',
        "candidate_model_source_sha256": None,
        "candidate_model_source": "embedded exact StockTransformer from baseline notebook",
        "train_table": TRAIN_TABLE,
        "infer_table": INFER_TABLE,
        "train_start": TRAIN_START,
        "train_end": TRAIN_END,
        "valid_start": VALID_START,
        "valid_end": VALID_END,
        "cache_root": str(cache_root),
    }
    logger.info("实验配置", **log_config)

    def initialize_online_cuda(seed):
        """在线上 runner 中延迟并显式初始化逻辑 cuda:0。

        关键点：
        - 在 DAI/memmap 阶段之前不触碰 CUDA runtime；
        - 只使用 CUDA_VISIBLE_DEVICES 映射后的逻辑 0 号卡；
        - 用最小分配和同步把异步错误固定在初始化阶段；
        - 只设置当前 GPU 的 RNG，不调用 manual_seed_all。
        """
        if not torch.cuda.is_available():
            raise RuntimeError(
                "线上环境未检测到可用 CUDA。"
                f" torch={torch.__version__},"
                f" torch.version.cuda={torch.version.cuda},"
                f" CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')}"
            )

        device_count = int(torch.cuda.device_count())
        if device_count < 1:
            raise RuntimeError(
                "torch.cuda.is_available() 为 True，但 device_count < 1。"
                f" CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')}"
            )

        # 在线平台可能把物理卡号重映射；进程内必须使用逻辑 cuda:0。
        torch.cuda.set_device(0)
        resolved = torch.device("cuda:0")

        try:
            torch.cuda.init()
            probe = torch.empty(32, dtype=torch.float32, device=resolved)
            probe.fill_(1.0)
            torch.cuda.synchronize(resolved)

            # 只设置当前可见 GPU，避免触碰其它在线设备/MIG 实例。
            torch.cuda.manual_seed(seed)
            del probe
            torch.cuda.synchronize(resolved)
        except Exception as exc:
            raise RuntimeError(
                "CUDA 在最小分配阶段失败；这不是模型或 DataLoader OOM。"
                f" torch={torch.__version__},"
                f" torch.version.cuda={torch.version.cuda},"
                f" CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')},"
                f" device_count={device_count}. "
                "若错误包含 forked subprocess/re-initialize CUDA，说明线上 runner "
                "在已初始化 CUDA 后 fork；需要平台使用 spawn 或干净 worker。"
            ) from exc

        logger.info(
            "CUDA 延迟初始化完成",
            device=str(resolved),
            current_device=int(torch.cuda.current_device()),
            device_count=device_count,
            device_name=torch.cuda.get_device_name(0),
            torch_version=torch.__version__,
            torch_cuda_build=torch.version.cuda,
            cuda_visible_devices=os.environ.get("CUDA_VISIBLE_DEVICES"),
        )
        return resolved

    # ---------- 模型: Transformer 编码 -> 可切换 Pooling -> 回归头 ----------
    class StockTransformer(nn.Module):
        def __init__(
            self,
            n_feat,
            d_model=128,
            nhead=4,
            nlayers=2,
            dim_ff=256,
            seq_len=SEQ_LEN,
            encoder_dropout=0.10,
            head_dropout=0.0,
            pooling="mean",
            input_field_dropout=0.0,
            gaussian_noise=0.0,
            causal_attention=False,
            pos_encoding="learned",
            head_type="linear",
        ):
            super().__init__()
            if d_model % nhead != 0:
                raise ValueError(f"d_model 必须能被 nhead 整除: d_model={d_model}, nhead={nhead}")
            self.pooling = pooling
            self.input_field_dropout = input_field_dropout
            self.gaussian_noise = gaussian_noise
            self.causal_attention = causal_attention
            self.pos_encoding = pos_encoding
            self.proj = nn.Linear(n_feat, d_model)
            if pos_encoding == "learned":
                self.pos = nn.Parameter(torch.zeros(1, seq_len, d_model))
            elif pos_encoding == "sinusoidal":
                self.register_buffer("pos", self._build_sinusoidal_pos(seq_len, d_model), persistent=False)
            else:
                raise ValueError(f"未知 position encoding: {pos_encoding}")

            layer = nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=dim_ff,
                dropout=encoder_dropout,
                batch_first=True,
                activation="gelu",
                norm_first=True,
            )
            self.encoder = nn.TransformerEncoder(layer, nlayers)

            if pooling in {"attn", "attn_recency"}:
                self.pool_score = nn.Sequential(
                    nn.Linear(d_model, d_model // 2),
                    nn.GELU(),
                    nn.Linear(d_model // 2, 1),
                )
            if pooling == "attn_recency":
                init_bias = torch.linspace(-0.5, 0.5, seq_len).view(1, seq_len, 1)
                self.recency = nn.Parameter(init_bias)

            head_dim = d_model * 2 if pooling == "mean_last" else d_model
            if head_type == "linear":
                self.head = nn.Sequential(
                    nn.LayerNorm(head_dim),
                    nn.Dropout(head_dropout),
                    nn.Linear(head_dim, 1),
                )
            elif head_type == "mlp":
                hidden = max(32, d_model // 2)
                self.head = nn.Sequential(
                    nn.LayerNorm(head_dim),
                    nn.Linear(head_dim, hidden),
                    nn.GELU(),
                    nn.Dropout(head_dropout),
                    nn.Linear(hidden, 1),
                )
            else:
                raise ValueError(f"未知 head type: {head_type}")

        @staticmethod
        def _build_sinusoidal_pos(seq_len, d_model):
            pos = torch.arange(seq_len, dtype=torch.float32).unsqueeze(1)
            div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
            pe = torch.zeros(1, seq_len, d_model, dtype=torch.float32)
            pe[0, :, 0::2] = torch.sin(pos * div)
            pe[0, :, 1::2] = torch.cos(pos * div[: pe[0, :, 1::2].shape[1]])
            return pe

        def _regularize_input(self, x):
            if self.training and self.input_field_dropout > 0:
                keep = 1.0 - self.input_field_dropout
                mask = (torch.rand(x.size(0), 1, x.size(2), device=x.device) < keep).float()
                x = x * mask / keep
            if self.training and self.gaussian_noise > 0:
                x = x + self.gaussian_noise * torch.randn_like(x)
            return x

        def _pool(self, h):
            if self.pooling == "mean":
                return h.mean(dim=1)
            if self.pooling == "last":
                return h[:, -1, :]
            if self.pooling == "mean_last":
                return torch.cat([h.mean(dim=1), h[:, -1, :]], dim=1)
            if self.pooling in {"attn", "attn_recency"}:
                logits = self.pool_score(h)
                if self.pooling == "attn_recency":
                    logits = logits + self.recency[:, : h.size(1), :]
                weight = torch.softmax(logits, dim=1)
                return (h * weight).sum(dim=1)
            raise ValueError(f"未知 pooling: {self.pooling}")

        def forward(self, x):
            x = self._regularize_input(x)
            token = self.proj(x) + self.pos[:, : x.size(1), :]
            attn_mask = None
            if self.causal_attention:
                attn_mask = torch.full((x.size(1), x.size(1)), float("-inf"), device=x.device)
                attn_mask = torch.triu(attn_mask, diagonal=1)
            h = self.encoder(token, mask=attn_mask)
            z = self._pool(h)
            return self.head(z).squeeze(-1)


    # ---------- 低内存数据层 ----------
    def pool(sd, ed):
        """区间内中证 1000 历史成分股代码。"""
        df = dai.query(
            "SELECT DISTINCT instrument FROM bigalpha_2026_instruments",
            filters={"date": [sd, ed]},
            compression=True,
        ).df()
        instruments = df["instrument"].astype(str).drop_duplicates().tolist()
        if not instruments:
            raise RuntimeError(f"pool 无标的: {sd}~{ed}")
        return instruments

    def _cache_signature(table, sd, ed, mode):
        payload = {
            "table": str(table),
            "sd": str(sd),
            "ed": str(ed),
            "mode": str(mode),
            "seq_len": SEQ_LEN,
            "field_set": FIELD_SET,
            "feature_cols": FEATURE_COLS,
            "vol_cols": VOL_COLS,
            "buffer_days": int(DATA_CONFIG["buffer_days"]),
            "instrument_chunk_size": int(DATA_CONFIG["instrument_chunk_size"]),
            "label_definition": "next_daily_raw_close/current_daily_raw_close-1",
            "input_preprocessing": "Baseline N0: volume log1p(clip>=0); inf/nan fill 0",
            "stats_method": "np_memmap_reshape_mean_std_float32",
        }
        raw = json.dumps(payload, ensure_ascii=False, sort_keys=True).encode("utf-8")
        return payload, hashlib.sha256(raw).hexdigest()

    def _read_manifest(root):
        path = Path(root) / "manifest.json"
        if not path.exists():
            raise FileNotFoundError(path)
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)

    def _log_manifest(root, split_name):
        manifest = _read_manifest(root)
        logger.info(
            f"{split_name} 缓存审计",
            samples=manifest["n_samples"],
            x_sha256=manifest["x_sha256"],
            key_sha256=manifest["key_sha256"],
            y_sha256=manifest.get("y_sha256"),
            label_quantiles=manifest.get("label_quantiles"),
            near_minus_one=manifest.get("near_minus_one"),
            first_keys=manifest.get("first_keys"),
            last_keys=manifest.get("last_keys"),
            signature_sha256=manifest["signature_sha256"],
            stats_sha256=manifest.get("stats_sha256"),
            stats_method=manifest.get("stats_method"),
        )
        return manifest

    def materialize_split(
        root,
        table,
        sd,
        ed,
        mode,
        instruments,
        train_stats=None,
    ):
        """将一个 split 顺序写入连续二进制文件。

        文件内容：
        X.dat          float32 [N, SEQ_LEN, N_FEAT]，仅做字段级 log/缺失填充，未标准化；
        y.dat          float32 [N]，train/valid 存在；
        date.dat       int64 [N]，自然日的 datetime64[ns]；
        instrument.dat S16 [N]；
        manifest.json  样本数、统计量、标签分布和 SHA-256 审计信息。
        """
        if mode not in {"train", "valid", "infer"}:
            raise ValueError(f"未知 mode: {mode}")

        root = Path(root)
        signature, signature_sha256 = _cache_signature(table, sd, ed, mode)

        manifest_path = root / "manifest.json"
        if (
            not DATA_CONFIG["rebuild_cache"]
            and manifest_path.exists()
        ):
            manifest = _read_manifest(root)
            if manifest.get("signature_sha256") != signature_sha256:
                raise RuntimeError(
                    f"{root} 缓存签名不一致。请设置 rebuild_cache=True 后重建。"
                )
            stats = train_stats
            if mode == "train":
                stats = (
                    np.asarray(manifest["feature_mean"], dtype=np.float32),
                    np.asarray(manifest["feature_std"], dtype=np.float32),
                )
            logger.info(f"{mode} 复用缓存", root=str(root), samples=manifest["n_samples"])
            return manifest, stats

        shutil.rmtree(root, ignore_errors=True)
        root.mkdir(parents=True, exist_ok=True)

        t0 = time.time()
        sd_ts = pd.Timestamp(sd)
        ed_ts = pd.Timestamp(ed)
        query_start = (
            sd_ts - pd.Timedelta(days=int(DATA_CONFIG["buffer_days"]))
        ).strftime("%Y-%m-%d %H:%M:%S")

        instruments = list(map(str, instruments))
        if MAX_TRAIN_INSTRUMENTS is not None:
            instruments = instruments[:MAX_TRAIN_INSTRUMENTS]

        write_buffer_samples = int(DATA_CONFIG["write_buffer_samples"])
        instrument_chunk_size = int(DATA_CONFIG["instrument_chunk_size"])

        x_buffer = np.empty(
            (write_buffer_samples, SEQ_LEN, N_FEAT),
            dtype=np.float32,
        )
        y_buffer = (
            np.empty(write_buffer_samples, dtype=np.float32)
            if mode in {"train", "valid"}
            else None
        )
        date_buffer = np.empty(write_buffer_samples, dtype=np.int64)
        instrument_buffer = np.empty(write_buffer_samples, dtype="S16")

        x_hasher = hashlib.sha256()
        y_hasher = hashlib.sha256() if y_buffer is not None else None
        key_hasher = hashlib.sha256()

        first_keys = []
        last_keys = deque(maxlen=10)
        n_samples = 0
        buffer_count = 0

        x_path = root / "X.dat"
        y_path = root / "y.dat"
        date_path = root / "date.dat"
        instrument_path = root / "instrument.dat"

        def flush_buffers(x_f, y_f, date_f, instrument_f):
            nonlocal buffer_count
            if buffer_count == 0:
                return

            x_block = x_buffer[:buffer_count]
            date_block = date_buffer[:buffer_count]
            instrument_block = instrument_buffer[:buffer_count]

            x_block.tofile(x_f)
            date_block.tofile(date_f)
            instrument_block.tofile(instrument_f)

            x_hasher.update(x_block.tobytes(order="C"))
            key_hasher.update(date_block.tobytes(order="C"))
            key_hasher.update(instrument_block.tobytes(order="C"))

            if y_buffer is not None:
                y_block = y_buffer[:buffer_count]
                y_block.tofile(y_f)
                y_hasher.update(y_block.tobytes(order="C"))

            buffer_count = 0

        with (
            x_path.open("wb") as x_f,
            date_path.open("wb") as date_f,
            instrument_path.open("wb") as instrument_f,
            y_path.open("wb") as y_f,
        ):
            total_chunks = math.ceil(len(instruments) / instrument_chunk_size)

            for chunk_no, start in enumerate(
                range(0, len(instruments), instrument_chunk_size),
                start=1,
            ):
                inst_chunk = instruments[start:start + instrument_chunk_size]

                sql = f"""
                    SELECT date, instrument, {", ".join(FEATURE_COLS)}
                    FROM {table}
                    ORDER BY instrument, date
                """
                df = dai.query(
                    sql,
                    filters={
                        "date": [query_start, str(ed)],
                        "instrument": inst_chunk,
                    },
                    compression=True,
                ).df()

                if df.empty:
                    logger.warning(
                        "DAI 股票块为空",
                        mode=mode,
                        chunk_no=chunk_no,
                        instruments=inst_chunk,
                    )
                    continue

                df["date"] = pd.to_datetime(df["date"])
                df["instrument"] = df["instrument"].astype(str)
                df.sort_values(["instrument", "date"], inplace=True)

                # 关键修复：任何输入变换之前保存原始 close。
                df["__raw_close_for_label"] = pd.to_numeric(
                    df["close"],
                    errors="coerce",
                )

                # 逐句对齐 0721 Baseline：仅数量字段 log1p，随后统一 inf/nan 填 0。
                for col in VOL_COLS:
                    df[col] = np.log1p(df[col].clip(lower=0))
                df[FEATURE_COLS] = (
                    df[FEATURE_COLS]
                    .replace([np.inf, -np.inf], np.nan)
                    .fillna(0.0)
                )

                for instrument, sub in df.groupby("instrument", sort=False):
                    if len(sub) <= SEQ_LEN:
                        continue

                    feats = sub[FEATURE_COLS].to_numpy(
                        dtype=np.float32,
                        copy=False,
                    )
                    raw_close = sub["__raw_close_for_label"].to_numpy(
                        dtype=np.float64,
                        copy=False,
                    )
                    bars_ns = sub["date"].to_numpy(dtype="datetime64[ns]")
                    days = bars_ns.astype("datetime64[D]")

                    close_pos = np.flatnonzero(
                        np.append(days[1:] != days[:-1], True)
                    )
                    close_px = raw_close[close_pos]
                    sample_days = days[close_pos]

                    for k, pos in enumerate(close_pos):
                        sample_date = pd.Timestamp(sample_days[k])

                        if sample_date < sd_ts or sample_date > ed_ts:
                            continue
                        if pos + 1 < SEQ_LEN:
                            continue

                        label = None
                        if mode in {"train", "valid"}:
                            if k + 1 < len(close_pos) and close_px[k] > 0:
                                ret = close_px[k + 1] / close_px[k] - 1.0
                                if np.isfinite(ret):
                                    label = np.float32(ret)

                            # 与 Baseline 相同：无法构造有限下一交易日收益时丢弃。
                            if label is None:
                                continue

                        x_buffer[buffer_count] = feats[
                            pos - SEQ_LEN + 1:pos + 1
                        ]
                        date_ns = np.datetime64(
                            sample_date.normalize().to_datetime64(),
                            "ns",
                        ).astype(np.int64)
                        instrument_bytes = str(instrument).encode("ascii")

                        date_buffer[buffer_count] = date_ns
                        instrument_buffer[buffer_count] = instrument_bytes
                        if y_buffer is not None:
                            y_buffer[buffer_count] = label

                        key_display = (
                            str(sample_date.date()),
                            str(instrument),
                        )
                        if len(first_keys) < 10:
                            first_keys.append(key_display)
                        last_keys.append(key_display)

                        buffer_count += 1
                        n_samples += 1

                        if buffer_count == write_buffer_samples:
                            flush_buffers(x_f, y_f, date_f, instrument_f)

                    del feats, raw_close, bars_ns, days, close_pos, close_px, sample_days

                del df
                gc.collect()

                if (
                    chunk_no == 1
                    or chunk_no == total_chunks
                    or chunk_no % 10 == 0
                ):
                    logger.info(
                        f"{mode} 缓存构建进度",
                        chunk=f"{chunk_no}/{total_chunks}",
                        samples=n_samples,
                        elapsed=round(time.time() - t0, 2),
                    )

            flush_buffers(x_f, y_f, date_f, instrument_f)

        if n_samples == 0:
            raise RuntimeError(
                f"materialize_split 无样本: mode={mode}, table={table}, {sd}~{ed}"
            )

        stats = train_stats
        manifest = {
            "signature": signature,
            "signature_sha256": signature_sha256,
            "mode": mode,
            "n_samples": int(n_samples),
            "seq_len": int(SEQ_LEN),
            "n_feat": int(N_FEAT),
            "feature_cols": list(FEATURE_COLS),
            "x_dtype": "float32",
            "y_dtype": "float32" if y_buffer is not None else None,
            "date_dtype": "int64",
            "instrument_dtype": "S16",
            "x_sha256": x_hasher.hexdigest(),
            "y_sha256": y_hasher.hexdigest() if y_hasher is not None else None,
            "key_sha256": key_hasher.hexdigest(),
            "first_keys": list(first_keys),
            "last_keys": list(last_keys),
            "elapsed_seconds": round(time.time() - t0, 3),
        }

        if mode == "train":
            # 严格复刻 Baseline 的 float32 归约路径。mean 可直接在完整 memmap 视图上
            # 计算；std 的全尺寸临时偏差数组也写入磁盘 scratch memmap，避免占用 RAM。
            x_mm = np.memmap(
                x_path,
                dtype=np.float32,
                mode="r",
                shape=(n_samples, SEQ_LEN, N_FEAT),
            )
            flat = x_mm.reshape(-1, N_FEAT)
            mean = flat.mean(0).astype(np.float32)

            scratch_path = root / "std_scratch.dat"
            squared_deviation = np.memmap(
                scratch_path,
                dtype=np.float32,
                mode="w+",
                shape=(n_samples, SEQ_LEN, N_FEAT),
            )
            np.subtract(
                x_mm,
                mean.reshape(1, 1, N_FEAT),
                out=squared_deviation,
            )
            np.multiply(
                squared_deviation,
                squared_deviation,
                out=squared_deviation,
            )
            var = squared_deviation.reshape(-1, N_FEAT).sum(0)
            np.true_divide(
                var,
                flat.shape[0],
                out=var,
                casting="unsafe",
            )
            np.sqrt(var, out=var)
            std = var.astype(np.float32) + 1e-6
            stats = mean, std

            stats_bytes = (
                np.ascontiguousarray(mean).tobytes()
                + np.ascontiguousarray(std).tobytes()
            )
            manifest["feature_mean"] = mean.tolist()
            manifest["feature_std"] = std.tolist()
            manifest["stats_sha256"] = hashlib.sha256(stats_bytes).hexdigest()
            manifest["stats_method"] = (
                "Baseline float32 mean/std reduction with disk scratch memmap"
            )
            del var, squared_deviation, flat, x_mm
            scratch_path.unlink(missing_ok=True)
        elif train_stats is None:
            raise ValueError(f"{mode} 必须传入训练集 stats")

        if y_buffer is not None:
            y_mm = np.memmap(
                y_path,
                dtype=np.float32,
                mode="r",
                shape=(n_samples,),
            )
            qs = np.percentile(
                y_mm,
                [0, 0.1, 1, 25, 50, 75, 99, 99.9, 100],
            )
            manifest["label_quantiles"] = {
                str(q): float(v)
                for q, v in zip(
                    [0, 0.1, 1, 25, 50, 75, 99, 99.9, 100],
                    qs,
                )
            }
            manifest["near_minus_one"] = int(np.sum(y_mm <= -0.999))
            manifest["finite_label_ratio"] = float(np.isfinite(y_mm).mean())
            del y_mm

        with manifest_path.open("w", encoding="utf-8") as f:
            json.dump(manifest, f, ensure_ascii=False, indent=2)

        logger.info(
            f"{mode} 缓存构建完成",
            root=str(root),
            samples=n_samples,
            elapsed=manifest["elapsed_seconds"],
        )
        return manifest, stats

    class MemmapWindowDataset(Dataset):
        """Map-style Dataset；每个进程按需打开只读 memmap。"""

        def __init__(self, root, return_y):
            self.root = Path(root)
            self.manifest = _read_manifest(self.root)
            self.n = int(self.manifest["n_samples"])
            self.seq_len = int(self.manifest["seq_len"])
            self.n_feat = int(self.manifest["n_feat"])
            self.return_y = bool(return_y)
            self._x = None
            self._y = None

            if self.return_y and self.manifest.get("y_dtype") is None:
                raise ValueError(f"{root} 不包含标签")

        def __len__(self):
            return self.n

        def _ensure_open(self):
            if self._x is None:
                self._x = np.memmap(
                    self.root / "X.dat",
                    dtype=np.float32,
                    mode="r",
                    shape=(self.n, self.seq_len, self.n_feat),
                )
            if self.return_y and self._y is None:
                self._y = np.memmap(
                    self.root / "y.dat",
                    dtype=np.float32,
                    mode="r",
                    shape=(self.n,),
                )

        def __getitem__(self, index):
            self._ensure_open()
            x = np.asarray(self._x[index], dtype=np.float32)
            if not self.return_y:
                return x
            return x, np.float32(self._y[index])

        def __getstate__(self):
            state = self.__dict__.copy()
            state["_x"] = None
            state["_y"] = None
            return state

    class NormalizeCollate:
        """只为当前 batch 分配连续数组，并原地标准化。"""

        def __init__(self, mean, std, return_y, label_clip=None):
            self.mean = np.asarray(mean, dtype=np.float32).reshape(1, 1, -1)
            self.std = np.asarray(std, dtype=np.float32).reshape(1, 1, -1)
            self.return_y = bool(return_y)
            self.label_clip = label_clip

        def __call__(self, batch):
            if self.return_y:
                xs, ys = zip(*batch)
            else:
                xs = batch

            x = np.stack(xs, axis=0).astype(np.float32, copy=False)
            np.subtract(x, self.mean, out=x)
            np.divide(x, self.std, out=x)
            x_tensor = torch.from_numpy(x)

            if not self.return_y:
                return x_tensor

            y = np.asarray(ys, dtype=np.float32)
            if self.label_clip is not None:
                lo, hi = self.label_clip
                np.clip(y, lo, hi, out=y)
            return x_tensor, torch.from_numpy(y)

    def make_memmap_loader(
        root,
        stats,
        return_y,
        shuffle,
        label_clip=None,
    ):
        dataset = MemmapWindowDataset(root, return_y=return_y)
        num_workers = int(DATA_CONFIG["num_workers"])

        kwargs = {
            "dataset": dataset,
            "batch_size": BATCH,
            "shuffle": bool(shuffle),
            "drop_last": False,
            "num_workers": num_workers,
            "pin_memory": device.type == "cuda",
            "collate_fn": NormalizeCollate(
                mean=stats[0],
                std=stats[1],
                return_y=return_y,
                label_clip=label_clip,
            ),
        }
        if num_workers > 0:
            kwargs["prefetch_factor"] = int(DATA_CONFIG["prefetch_factor"])
            kwargs["persistent_workers"] = True

        return DataLoader(**kwargs)

    def load_labels(root):
        manifest = _read_manifest(root)
        return np.memmap(
            Path(root) / "y.dat",
            dtype=np.float32,
            mode="r",
            shape=(int(manifest["n_samples"]),),
        )

    def load_index_frame(root):
        manifest = _read_manifest(root)
        n = int(manifest["n_samples"])

        date_mm = np.memmap(
            Path(root) / "date.dat",
            dtype=np.int64,
            mode="r",
            shape=(n,),
        )
        instrument_mm = np.memmap(
            Path(root) / "instrument.dat",
            dtype="S16",
            mode="r",
            shape=(n,),
        )

        dates = pd.to_datetime(np.asarray(date_mm, dtype=np.int64))
        instruments = np.char.decode(
            np.asarray(instrument_mm, dtype="S16"),
            "ascii",
        )
        result = pd.DataFrame(
            {
                "date": dates.normalize(),
                "instrument": instruments.astype(str),
            }
        )
        del date_mm, instrument_mm
        return result

    def evaluate_loader(model, loader):
        model.eval()
        preds, losses = [], []
        loss_fn = nn.MSELoss()

        with torch.no_grad():
            for xb, yb in loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)
                pred = model(xb)
                losses.append(loss_fn(pred, yb).item())
                preds.append(pred.cpu().numpy())

        return (
            float(np.mean(losses)),
            np.concatenate(preds).astype(np.float32),
        )

    def _daily_winsor_zscore(s, lower_q=0.025, upper_q=0.975):
        s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
        valid = s.dropna()
        if len(valid) < 5:
            return s * np.nan
        lo, hi = valid.quantile([lower_q, upper_q])
        clipped = s.clip(lo, hi)
        mu = clipped.mean()
        sd = clipped.std(ddof=0)
        if not np.isfinite(sd) or sd < 1e-12:
            return clipped * 0.0
        return (clipped - mu) / sd

    def _daily_long_short_return(g, score_col="score_z", y_col="y", q=0.2):
        g = g[[score_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna()
        if len(g) < 20 or g[score_col].nunique() <= 1:
            return np.nan
        n = max(1, int(len(g) * q))
        ordered = g.sort_values(score_col)
        short_ret = ordered.iloc[:n][y_col].mean()
        long_ret = ordered.iloc[-n:][y_col].mean()
        return float(long_ret - short_ret)

    def _summarize_daily_ic(daily_ic, prefix="valid"):
        ic_arr = np.asarray(daily_ic, dtype=np.float64)
        ic_mean = float(ic_arr.mean()) if len(ic_arr) else 0.0
        ic_std = float(ic_arr.std(ddof=1)) if len(ic_arr) > 1 else 0.0
        ic_ir = float(ic_mean / (ic_std + 1e-12)) if len(ic_arr) > 1 else 0.0
        return {
            f"{prefix}_ic_mean": ic_mean,
            f"{prefix}_ic_ir": ic_ir,
            f"{prefix}_days": int(len(ic_arr)),
        }

    def load_eval_pool(sd, ed):
        stk = dai.query(
            "SELECT date, instrument FROM bigalpha_2026_instruments",
            filters={"date": [sd, ed]},
            compression=True,
        ).df()
        if stk.empty:
            raise RuntimeError(f"验证成分池为空: {sd}~{ed}")
        stk["date"] = pd.to_datetime(stk["date"]).dt.normalize()
        stk["instrument"] = stk["instrument"].astype(str)
        return stk.drop_duplicates(["date", "instrument"])

    def calc_raw_ic_metrics(pred, y_raw, idx_df):
        tmp = idx_df.copy()
        tmp["pred"] = pred.astype(np.float64)
        tmp["y"] = y_raw.astype(np.float64)
        daily_ic, daily_rank_ic = [], []
        for _, g in tmp.groupby("date"):
            if len(g) < 5 or g["pred"].nunique() <= 1 or g["y"].nunique() <= 1:
                continue
            ic = g["pred"].corr(g["y"], method="pearson")
            ric = g["pred"].corr(g["y"], method="spearman")
            if np.isfinite(ic):
                daily_ic.append(ic)
            if np.isfinite(ric):
                daily_rank_ic.append(ric)
        metrics = _summarize_daily_ic(daily_ic, prefix="raw_valid")
        ric_arr = np.asarray(daily_rank_ic, dtype=np.float64)
        metrics["raw_valid_rank_ic_mean"] = float(ric_arr.mean()) if len(ric_arr) else 0.0
        return metrics

    def calc_official_like_metrics(pred, y_raw, idx_df, eval_pool_df):
        tmp = idx_df.copy()
        tmp["date"] = pd.to_datetime(tmp["date"]).dt.normalize()
        tmp["instrument"] = tmp["instrument"].astype(str)
        tmp["score"] = pred.astype(np.float64)
        tmp["y"] = y_raw.astype(np.float64)
        tmp = tmp.replace([np.inf, -np.inf], np.nan).dropna(subset=["score", "y"])
        tmp = tmp.drop_duplicates(["date", "instrument"], keep="last")
        tmp = pd.merge(tmp, eval_pool_df, on=["date", "instrument"], how="inner")
        if tmp.empty:
            raise RuntimeError("official-like valid 无样本: 成分池对齐后为空")

        tmp["score_z"] = tmp.groupby("date")["score"].transform(_daily_winsor_zscore)
        tmp = tmp.replace([np.inf, -np.inf], np.nan).dropna(subset=["score_z", "y"])

        daily_ic, daily_rank_ic, daily_ls = [], [], []
        coverage, raw_dispersion, dispersion = [], [], []
        for d, g in tmp.groupby("date"):
            if len(g) < 20 or g["score_z"].nunique() <= 1 or g["y"].nunique() <= 1:
                continue
            ic = g["score_z"].corr(g["y"], method="pearson")
            ric = g["score_z"].corr(g["y"], method="spearman")
            if np.isfinite(ic):
                daily_ic.append(ic)
                coverage.append(len(g))
                raw_dispersion.append(float(g["score"].std(ddof=0)))
                dispersion.append(float(g["score_z"].std(ddof=0)))
            if np.isfinite(ric):
                daily_rank_ic.append(ric)
            ls = _daily_long_short_return(g)
            if np.isfinite(ls):
                daily_ls.append(ls)

        metrics = _summarize_daily_ic(daily_ic, prefix="valid")
        ric_arr = np.asarray(daily_rank_ic, dtype=np.float64)
        ls_arr = np.asarray(daily_ls, dtype=np.float64)
        metrics.update({
            "valid_rank_ic_mean": float(ric_arr.mean()) if len(ric_arr) else 0.0,
            "valid_long_short_mean": float(ls_arr.mean()) if len(ls_arr) else 0.0,
            "valid_long_short_sharpe": float(ls_arr.mean() / (ls_arr.std(ddof=1) + 1e-12) * math.sqrt(240)) if len(ls_arr) > 1 else 0.0,
            "valid_stress_proxy_ic_ir": float(np.mean(np.sort(np.asarray(daily_ic, dtype=np.float64))[:max(1, int(len(daily_ic) * 0.2))]) / (np.std(np.sort(np.asarray(daily_ic, dtype=np.float64))[:max(1, int(len(daily_ic) * 0.2))], ddof=1) + 1e-12)) if len(daily_ic) > 5 else 0.0,
            "valid_avg_coverage": float(np.mean(coverage)) if coverage else 0.0,
            "valid_raw_score_dispersion": float(np.mean(raw_dispersion)) if raw_dispersion else 0.0,
            "valid_score_dispersion": float(np.mean(dispersion)) if dispersion else 0.0,
        })
        return metrics

    def plot_valid_loss(history, best_epoch=None):
        hist = pd.DataFrame(history)
        try:
            import matplotlib.pyplot as plt
            ax = hist.plot(x="epoch", y="valid_score", marker="o", legend=False)
            if best_epoch is not None:
                ax.axvline(best_epoch, color="red", linestyle="--", alpha=0.7, label="best epoch")
                ax.legend()
            ax.set_title("2023H2 official-like early-stop score")
            ax.set_xlabel("epoch")
            ax.set_ylabel("official-like IC mean + 0.1 * IC IR")
            plt.show()
            ax = hist.plot(x="epoch", y="valid_mse", marker="o", legend=False)
            if best_epoch is not None:
                ax.axvline(best_epoch, color="red", linestyle="--", alpha=0.7, label="best epoch")
                ax.legend()
            ax.set_title("2023H2 valid MSE reference")
            ax.set_xlabel("epoch")
            ax.set_ylabel("MSE")
            plt.show()
            ax = hist.plot(x="epoch", y=["train_mse", "valid_mse"], marker="o")
            if best_epoch is not None:
                ax.axvline(best_epoch, color="red", linestyle="--", alpha=0.7, label="best epoch")
                ax.legend()
            ax.set_title("train MSE vs 2023H2 valid MSE")
            ax.set_xlabel("epoch")
            ax.set_ylabel("MSE")
            plt.show()
        except Exception as e:
            logger.warning("valid metric 作图失败", error=str(e))
        return hist

# ---------- 训练 + 2023 后半年验证 ----------
    logger.info(
        "构建训练/验证缓存",
        train_start=TRAIN_START,
        train_end=TRAIN_END,
        valid_start=VALID_START,
        valid_end=VALID_END,
    )

    if DATA_CONFIG["rebuild_cache"]:
        shutil.rmtree(cache_root, ignore_errors=True)
    cache_root.mkdir(parents=True, exist_ok=True)

    train_root = cache_root / "train"
    valid_root = cache_root / "valid"
    infer_root = cache_root / "infer"

    train_instruments = pool(TRAIN_START, TRAIN_END)
    valid_instruments = pool(VALID_START, VALID_END)
    instruments = sorted(set(train_instruments) | set(valid_instruments))
    if MAX_TRAIN_INSTRUMENTS is not None:
        instruments = instruments[:MAX_TRAIN_INSTRUMENTS]

    train_manifest, stats = materialize_split(
        root=train_root,
        table=TRAIN_TABLE,
        sd=TRAIN_START,
        ed=TRAIN_END,
        mode="train",
        instruments=instruments,
        train_stats=None,
    )
    valid_manifest, _ = materialize_split(
        root=valid_root,
        table=TRAIN_TABLE,
        sd=VALID_START,
        ed=VALID_END,
        mode="valid",
        instruments=instruments,
        train_stats=stats,
    )

    _log_manifest(train_root, "train")
    _log_manifest(valid_root, "valid")

    ytr_raw = load_labels(train_root)
    yva_raw = load_labels(valid_root)
    idx_va = load_index_frame(valid_root)

    label_lo, label_hi = np.percentile(ytr_raw, [1, 99])
    label_clip = (float(label_lo), float(label_hi))

    logger.info(
        "标签截尾阈值",
        label_lo=label_clip[0],
        label_hi=label_clip[1],
        train_near_minus_one=int(np.sum(ytr_raw <= -0.999)),
        valid_near_minus_one=int(np.sum(yva_raw <= -0.999)),
    )

    valid_eval_pool = load_eval_pool(VALID_START, VALID_END)
    logger.info(
        "A1 验证口径",
        pool_rows=len(valid_eval_pool),
        pool_days=valid_eval_pool["date"].nunique(),
        preprocessing="pool_align + daily_winsor_2.5_97.5 + daily_zscore",
        barra_neutralization=False,
    )

    # 所有 DAI 查询和 memmap 物化完成后，才初始化 CUDA。
    device = initialize_online_cuda(SEED)

    model = StockTransformer(
        N_FEAT,
        d_model=EXPERIMENT_CONFIG["d_model"],
        nhead=EXPERIMENT_CONFIG["nhead"],
        nlayers=EXPERIMENT_CONFIG["nlayers"],
        dim_ff=EXPERIMENT_CONFIG["dim_ff"],
        seq_len=SEQ_LEN,
        encoder_dropout=EXPERIMENT_CONFIG["encoder_dropout"],
        head_dropout=EXPERIMENT_CONFIG["head_dropout"],
        pooling=EXPERIMENT_CONFIG["pooling"],
        input_field_dropout=EXPERIMENT_CONFIG["input_field_dropout"],
        gaussian_noise=EXPERIMENT_CONFIG["gaussian_noise"],
        causal_attention=EXPERIMENT_CONFIG["causal_attention"],
        pos_encoding=EXPERIMENT_CONFIG["pos_encoding"],
        head_type=EXPERIMENT_CONFIG["head_type"],
    )

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(
        "CPU 模型构造完成",
        n_params=n_params,
        parameter_devices=sorted({str(p.device) for p in model.parameters()}),
    )

    try:
        model = model.to(device, non_blocking=False)
        torch.cuda.synchronize(device)
    except Exception as exc:
        raise RuntimeError(
            "最小 CUDA 探针已通过，但模型迁移失败。"
            f" device={device},"
            f" allocated={torch.cuda.memory_allocated(0)},"
            f" reserved={torch.cuda.memory_reserved(0)}"
        ) from exc

    logger.info("可训练参数量", n_params=n_params)
    assert n_params == EXPERIMENT_CONFIG["expected_parameters"], f"K00 exact 参数量失败: got={n_params}, expected={EXPERIMENT_CONFIG['expected_parameters']}"

    train_loader = make_memmap_loader(
        train_root,
        stats=stats,
        return_y=True,
        shuffle=True,
        label_clip=label_clip,
    )
    valid_loader = make_memmap_loader(
        valid_root,
        stats=stats,
        return_y=True,
        shuffle=False,
        label_clip=label_clip,
    )

    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()
    rdrop_lambda = float(EXPERIMENT_CONFIG["rdrop_lambda"])

    history = []
    best_state = None
    best_score = -math.inf
    best_epoch = 0
    bad_epochs = 0

    for ep in range(EPOCHS):
        t0 = time.time()
        total_loss = 0.0
        nb = 0
        model.train()

        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad()
            if rdrop_lambda > 0:
                pred1 = model(xb)
                pred2 = model(xb)
                loss = (
                    0.5 * loss_fn(pred1, yb)
                    + 0.5 * loss_fn(pred2, yb)
                    + rdrop_lambda * loss_fn(pred1, pred2)
                )
            else:
                loss = loss_fn(model(xb), yb)

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += float(loss.item())
            nb += 1

        train_mse = total_loss / max(nb, 1)
        valid_mse, valid_pred = evaluate_loader(model, valid_loader)

        ic_metrics = calc_official_like_metrics(
            valid_pred,
            yva_raw,
            idx_va,
            valid_eval_pool,
        )
        ic_metrics.update(
            calc_raw_ic_metrics(
                valid_pred,
                yva_raw,
                idx_va,
            )
        )

        early_stop_metric = EXPERIMENT_CONFIG.get(
            "early_stop_metric",
            "official_ic_plus_ir",
        )
        if early_stop_metric == "official_ic_plus_ir":
            valid_score = (
                ic_metrics["valid_ic_mean"]
                + 0.1 * ic_metrics["valid_ic_ir"]
            )
        elif early_stop_metric == "raw_ic_plus_ir":
            valid_score = (
                ic_metrics["raw_valid_ic_mean"]
                + 0.1 * ic_metrics["raw_valid_ic_ir"]
            )
        elif early_stop_metric == "rank_ic":
            valid_score = ic_metrics["valid_rank_ic_mean"]
        elif early_stop_metric == "long_short_sharpe":
            valid_score = ic_metrics["valid_long_short_sharpe"]
        else:
            raise ValueError(
                f"未知 early_stop_metric: {early_stop_metric}"
            )

        row = {
            "epoch": ep + 1,
            "train_mse": float(train_mse),
            "valid_mse": float(valid_mse),
            "valid_score": float(valid_score),
            **ic_metrics,
        }
        history.append(row)
        logger.info(
            "epoch 完成",
            elapsed=round(time.time() - t0, 2),
            **row,
        )

        if valid_score > best_score + MIN_DELTA:
            best_score = valid_score
            best_epoch = ep + 1
            # 只改变 checkpoint 存储设备，不改变模型、RNG 或优化轨迹。
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                logger.info(
                    "early stopping",
                    best_epoch=best_epoch,
                    best_score=best_score,
                )
                break

    history_df = plot_valid_loss(history, best_epoch)
    best_row = (
        history_df.loc[history_df["epoch"] == best_epoch]
        .iloc[0]
        .to_dict()
    )
    logger.info(
        "训练完成",
        best_epoch=best_epoch,
        best_score=best_score,
        early_stop_metric=EXPERIMENT_CONFIG.get(
            "early_stop_metric",
            "official_ic_plus_ir",
        ),
        best_valid_mse=float(best_row["valid_mse"]),
        best_valid_ic_mean=float(best_row["valid_ic_mean"]),
        best_valid_ic_ir=float(best_row["valid_ic_ir"]),
        best_valid_rank_ic_mean=float(
            best_row.get("valid_rank_ic_mean", 0.0)
        ),
        best_valid_long_short_sharpe=float(
            best_row.get("valid_long_short_sharpe", 0.0)
        ),
        best_valid_stress_proxy_ic_ir=float(
            best_row.get("valid_stress_proxy_ic_ir", 0.0)
        ),
        best_valid_raw_score_dispersion=float(
            best_row.get("valid_raw_score_dispersion", 0.0)
        ),
        best_raw_valid_ic_mean=float(
            best_row.get("raw_valid_ic_mean", 0.0)
        ),
        best_raw_valid_ic_ir=float(
            best_row.get("raw_valid_ic_ir", 0.0)
        ),
        best_raw_valid_rank_ic_mean=float(
            best_row.get("raw_valid_rank_ic_mean", 0.0)
        ),
    )
    if best_state is not None:
        model.load_state_dict(best_state)

    # ---------- 推理 ----------
    logger.info(
        "构建测试缓存并预测",
        table=INFER_TABLE,
        start=str(start_date),
        end=str(end_date),
    )

    infer_instruments = pool(start_date, end_date)
    infer_manifest, _ = materialize_split(
        root=infer_root,
        table=INFER_TABLE,
        sd=start_date,
        ed=end_date,
        mode="infer",
        instruments=infer_instruments,
        train_stats=stats,
    )
    _log_manifest(infer_root, "infer")

    idx_df = load_index_frame(infer_root)
    infer_loader = make_memmap_loader(
        infer_root,
        stats=stats,
        return_y=False,
        shuffle=False,
        label_clip=None,
    )

    model.eval()
    preds = []
    with torch.no_grad():
        for xb in infer_loader:
            xb = xb.to(device, non_blocking=True)
            preds.append(model(xb).cpu().numpy())

    pred_arr = np.concatenate(preds).astype(np.float64)
    if len(pred_arr) != len(idx_df):
        raise RuntimeError(
            f"推理长度不一致: pred={len(pred_arr)}, index={len(idx_df)}"
        )
    idx_df["score"] = pred_arr

    # ---------- 对齐中证 1000 + 规范输出 ----------
    stk = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()
    stk["date"] = pd.to_datetime(stk["date"]).dt.normalize()
    stk["instrument"] = stk["instrument"].astype(str)

    result = (
        pd.merge(
            idx_df,
            stk,
            on=["date", "instrument"],
            how="inner",
        )
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["score"])
        .drop_duplicates(["date", "instrument"])
        [["date", "instrument", "score"]]
        .reset_index(drop=True)
    )

    logger.info(
        "分数构建完成",
        rows=len(result),
        days=result["date"].nunique(),
        instruments=result["instrument"].nunique(),
    )

    if DATA_CONFIG["cleanup_cache_at_end"]:
        del train_loader, valid_loader, infer_loader
        del ytr_raw, yva_raw
        gc.collect()
        shutil.rmtree(cache_root, ignore_errors=True)
        logger.info("已清理磁盘缓存", cache_root=str(cache_root))

    return result


if __name__ == "__main__":
    from bigmodule import M
    import structlog

    logger = structlog.get_logger()

    datasources = {
        "bar1m": "bigalpha_2026_stock_bar1m",
    }

    start_date = "2024-01-01 00:00:00"
    end_date = "2024-12-31 23:59:59"

    logger.info(
        "计算分数",
        start=start_date,
        end=end_date,
    )
    score_data = main(
        datasources,
        start_date,
        end_date,
    )
    print(score_data.head())

    logger.info("开始评估分数")
    result = M.bigalpha_eval._latest(
        factor_data=score_data,
        start_date=start_date,
        end_date=end_date,
        show=True,
    )